# Домашнее задание 5. Оптимизация с равенствами

**Максимум:** 10 баллов (+2 бонус). **Время:** ориентировочно 2,5–3 часа, бонус — ещё 20–30 минут.

Три независимые истории: эллипс (условия оптимальности), цилиндрическая банка (Ньютон–Лагранж), калибровка робота по измерениям (обобщённый Гаусс–Ньютон). Все необходимые математические инструменты есть в лекциях 8–9.

**Как сдавать.** Заполните ячейки с `# TODO` и ответы `_Ваш ответ:_ TODO`. Выполните ноутбук целиком (Restart Kernel → Run All), сохраните результаты и сдайте как `hw5/homework.ipynb` в вашем закрытом `numopt-<логин>` организации `optimisation-misis-2026`. Инструкция: [CONTRIBUTING.md](https://github.com/IlyaChichkanov/Numerical-Optimization/blob/main/CONTRIBUTING.md). Решения пишите самостоятельно. Числа округляйте разумно, объясняйте по результатам.

**Материал:** [лекция 8](https://github.com/IlyaChichkanov/Numerical-Optimization/blob/main/lectures/lecture08/lecture08.md) — Лагранж, LICQ, редуцированный гессиан; [лекция 9](https://github.com/IlyaChichkanov/Numerical-Optimization/blob/main/lectures/lecture09/lecture09.md) — ККТ, Ньютон–Лагранж и обобщённый Гаусс–Ньютон.

| Задача | Баллы | Основной результат |
|---|---:|---|
| 1. Эллипс | 2 | Найти и классифицировать стационарные точки |
| 2. Банка | 3 | Реализовать один алгоритм Ньютона–Лагранжа |
| 3. Калибровка | 5 | Реализовать замену гессиана Гаусса–Ньютона и исследовать измерения |
| Бонус | +2 | Две причины, почему прямой Ньютон может подвести |

Нужны NumPy и Matplotlib. Производные в программных задачах и функции для построения рисунков уже даны.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

BLUE, ORANGE, AQUA, RED, VIOLET, GRAY = "#2a78d6", "#eb6834", "#1baf7a", "#e34948", "#4a3aa7", "#8a8985"
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .2,
                     "axes.spines.top": False, "axes.spines.right": False, "legend.frameon": False,
                     "axes.prop_cycle": plt.cycler(color=[BLUE, ORANGE, AQUA, VIOLET])})

def plot_ellipse():
    t = np.linspace(0, 2*np.pi, 300)
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.plot(2*np.cos(t), np.sin(t), color=BLUE)
    ax.set(xlabel="x", ylabel="y", title="Допустимое множество: эллипс")
    ax.axis("equal")
    plt.tight_layout()
    return ax

def ellipse_kkt(x, lam):
    """f=y, g=(x²+4y²-4)/2; знак L=f-lambda*g."""
    u, v = np.asarray(x, float)
    a = float(lam[0])
    return (np.array([0., 1.]), np.array([[u, 4*v]]),
            -a*np.diag([1., 4.]), np.array([(u*u+4*v*v-4)/2]))

def can_kkt(x, lam):
    """f=r²+rh, g=r²h-2, L=f-lambda*g."""
    r, h = np.asarray(x, float)
    a = float(lam[0])
    return (np.array([2*r+h, r]),
            np.array([[2*r*h, r*r]]),
            np.array([[2-2*a*h, 1-2*a*r], [1-2*a*r, 0.]]),
            np.array([r*r*h-2]))

def plot_histories(histories, labels, title):
    fig, ax = plt.subplots(figsize=(7, 3.5))
    for hist, label in zip(histories, labels):
        ax.semilogy([max(float(a[2]), 1e-16) for a in hist], "-o", ms=4, label=label)
    ax.set(xlabel="Номер итерации", ylabel="Норма невязки ККТ", title=title)
    ax.legend()
    plt.tight_layout()
    return ax

# Ориентиры в локальной системе робота. Координаты намеренно не симметричны.
POINTS = np.array([[0., 0.], [1., 0.], [0., 1.], [2., 1.], [-1., 1.], [1.5, -.5]])
TRUE_ANGLE = .65
TRUE_SHIFT = np.array([.9, -.35])

def make_measurements(sigma=.035):
    phi = TRUE_ANGLE
    R = np.array([[np.cos(phi), -np.sin(phi)], [np.sin(phi), np.cos(phi)]])
    rng = np.random.default_rng(2026)
    return (R @ POINTS.T).T + TRUE_SHIFT + rng.normal(0, sigma, POINTS.shape)

def design_block(p):
    """A_i @ (c,s,t_x,t_y) = R(c,s) p + t."""
    px, py = p
    return np.array([[px, -py, 1., 0.], [py, px, 0., 1.]])

A = np.vstack([design_block(p) for p in POINTS])

def calibration_kkt(measured):
    """Возвращает (grad f, J, Hessian L, g) для 1/2||A z-b||² и (c²+s²-1)/2=0."""
    b = np.asarray(measured, float).reshape(-1)
    G, a = A.T @ A, A.T @ b
    D = np.diag([1., 1., 0., 0.])
    def kkt(z, lam):
        z = np.asarray(z, float)
        return (G @ z - a, np.array([[z[0], z[1], 0., 0.]]),
                G - float(lam[0]) * D,
                np.array([.5*(z[0]**2+z[1]**2-1)]))
    return kkt

def plot_calibration(measured, z):
    pred = (A @ z).reshape(-1, 2)
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.scatter(measured[:, 0], measured[:, 1], marker="x", s=60, color=RED, label="измерения")
    ax.scatter(pred[:, 0], pred[:, 1], s=40, color=BLUE, label="модель")
    for a, b in zip(measured, pred):
        ax.plot([a[0], b[0]], [a[1], b[1]], color=GRAY, lw=.9)
    ax.set(xlabel="X", ylabel="Y", title="Калибровка: измерения и модель")
    ax.axis("equal")
    ax.legend()
    plt.tight_layout()
    return ax

print("Готово: матрица измерений A имеет форму", A.shape)

## Задача 1. Две стационарные точки (2 балла, 20–25 минут)

Нужно минимизировать высоту точки на эллипсе:

$$
\min_{x,y} f(x,y)=y,\qquad g(x,y)=\tfrac12(x^2+4y^2-4)=0.
$$

Используйте соглашение курса $L=f-\lambda g$.

**(а) 1 балл.** Выпишите $\nabla f$, $\nabla g$, условия Лагранжа и найдите **все** стационарные точки вместе с множителями.

**(б) 1 балл.** Проверьте LICQ в обеих точках. Найдите касательный вектор $Z$ единичной длины и число $Z^\top\nabla^2_{xx}LZ$ в каждой точке. Какие точки являются строгим локальным минимумом и максимумом? Сверьтесь с графиком.

*Полный балл:* расчёт обеих точек, LICQ и знак редуцированного гессиана с выводом. *Половина:* правильные точки без проверки условий.

In [ ]:
plot_ellipse(); plt.show()

_Ваш ответ:_ TODO

## Задача 2. Банка и Ньютон–Лагранж (3 балла, 45–55 минут)

Закрытая цилиндрическая банка объёма $2\pi$ имеет радиус $r>0$ и высоту $h>0$. Её площадь поверхности равна $2\pi(r^2+rh)$. Сократим постоянный множитель:

$$
\min_{r,h>0} f(r,h)=r^2+rh,\qquad g(r,h)=r^2h-2=0.
$$

Метод пишем общим — для любого числа переменных $n$ и ограничений $m$: он понадобится и в задаче 3. Равенство не следит за знаком $r$ и $h$, поэтому промежуточные точки могут оказаться с $r\le0$ или $h\le0$.

**(а) 1,5 балла.** Напишите общую функцию `newton_lagrange(kkt, x0, lam0, tol=1e-10, maxit=50)`. На каждом шаге `kkt(x, lam)` возвращает $(\nabla f,J,H,g)$, где $H=\nabla^2L$. Решите

$$
\begin{pmatrix}H&-J^\top\\J&0\end{pmatrix}
\begin{pmatrix}p\\\Delta\lambda\end{pmatrix}
=-\begin{pmatrix}\nabla f-J^\top\lambda\\g\end{pmatrix}
$$

и обновите $x,\lambda$. Верните **список** $(x_k,\lambda_k,\|F_k\|)$ с начальной точкой; при $\|F_k\|\leq\mathrm{tol}$ или $k=\mathrm{maxit}$ закончите.

**(б) 1,5 балла.** Найдите оптимальную банку аналитически: подставьте $h=2/r^2$ в $f$. Затем сделайте три запуска метода:

- из $(r,h)=(1.1,\ 1.7)$ с $\lambda^{(0)}=0$;
- из $(1.5,\ 1.0)$ с $\lambda^{(0)}=0$;
- из $(1.5,\ 1.0)$ со стартовым множителем по МНК, как в лекции 9 §2.4: $\lambda^{(0)}$ — решение задачи $\min_\lambda\lVert\nabla f(x^{(0)})-J(x^{(0)})^\top\lambda\rVert$, одна строка `np.linalg.lstsq`.

Для каждого запуска выведите число шагов, ответ, значение $g$ в ответе и остались ли все точки с $r,h>0$. Нарисуйте `plot_histories` для трёх запусков. Почему из $(1.5,\ 1.0)$ при $\lambda^{(0)}=0$ нужно намного больше шагов, чем с множителем по МНК? Посмотрите, куда ведёт первый шаг.

*Полный балл:* работающий общий алгоритм, аналитический ответ, три запуска, график и объяснение через стартовый множитель. *Половина:* правильные числа без объяснения.

In [ ]:
def newton_lagrange(kkt, x0, lam0, tol=1e-10, maxit=50):
    # TODO: ваш код
    raise NotImplementedError

In [ ]:
# Самопроверка общего метода: аналитический ответ банки — (r,h,lambda)=(1,2,1).
test = newton_lagrange(can_kkt, [1.1, 1.7], [0.])
assert np.allclose(test[-1][0], [1., 2.], atol=1e-8), "Проверьте знаки в матрице ККТ"
assert np.allclose(test[-1][1], [1.], atol=1e-8) and test[-1][2] < 1e-8
print("Ньютон–Лагранж: проверка пройдена")

In [ ]:
# TODO: ваш код
# Три запуска, вывод чисел и график plot_histories(...)


_Ваш ответ:_ TODO

## Задача 3. Калибровка поворота и переноса (5 баллов, около 60 минут)

У робота известны координаты шести ориентиров $p_i\in\mathbb R^2$ в его собственной системе координат. Камера измерила их глобальные положения $q_i$ **с шумом**. Нужно найти поворот и перенос, которые лучше всего объясняют измерения.

Параметризуем поворот числами $c,s$, а перенос — $(t_x,t_y)$. Переменные $z=(c,s,t_x,t_y)\in\mathbb R^4$:

$$
R(c,s)=\begin{pmatrix}c&-s\\s&c\end{pmatrix},\quad
g(z)=\tfrac12(c^2+s^2-1)=0,
$$

$$
\min_z f(z)=\frac12\sum_{i=1}^6\|R(c,s)p_i+t-q_i\|^2.
$$

Данные фиксированы генератором `make_measurements(sigma=.035)`. Готовая матрица `A` имеет 12 строк и 4 столбца, так что вектор невязок $r(z)=Az-b$, где $b$ — все координаты $q_i$ подряд. Функция `calibration_kkt(measured)` возвращает производные и ограничение для Ньютона–Лагранжа; `A` — якобиан $J_r$.

**(а) 2 балла — постановка и алгоритм.** Для точки $p=(p_x,p_y)$ выпишите матрицу $A_i$ размера $2\times4$, для которой $R(c,s)p+t=A_i z$. Чему равен $\nabla^2 f$? Почему $\nabla^2L$ отличается от $\nabla^2 f$? Каков размер матрицы ККТ? Затем напишите `gn_kkt(kkt, jac_r)`: обёртку, которая возвращает те же $\nabla f$, $J$, $g$, но матрицу $H_{\mathrm{GN}}=J_r^\top J_r$ вместо точного гессиана. Цикл метода не переписывайте: `newton_lagrange(gn_kkt(kkt, jac_r), z0, lam0)` из задачи 2 и есть обобщённый Гаусс–Ньютон.

**(б) 3 балла — эксперимент.** Возьмите шум $\sigma=0.035$, затем $\sigma=0.6$ (seed тот же). В каждом случае запустите оба метода из $z_0=(1,0,0,0)$, $\lambda_0=0$. Выведите число итераций, угол $\varphi=\operatorname{atan2}(s,c)$, перенос, $\lVert r(z^\ast)\rVert$ и $\lambda^\ast$. Нарисуйте `plot_histories` для двух методов при $\sigma=0.035$ и `plot_calibration` для найденного преобразования.

**Объясните:** почему оптимальная невязка не равна нулю; что выбрасывает Гаусс–Ньютон; почему здесь он почти не уступает Ньютону–Лагранжу. Для последнего сравните $\lambda^\ast$ с собственными числами $A^\top A$ (`np.linalg.eigvalsh(A.T @ A)`).

*Полный балл:* (а) — формула $A_i$, оба гессиана, размер $K$, верная `gn_kkt`; (б) — числа для двух шумов, два графика и объяснение через $\lambda^\ast$ и собственные числа $A^\top A$. *Половина:* численные результаты без пояснений.

_Ваш ответ:_ TODO

In [ ]:
def gn_kkt(kkt, jac_r):
    # TODO: ваш код
    raise NotImplementedError

In [ ]:
# Проверка обёртки: изменяется только гессиан, остальные блоки остаются теми же.
sample_kkt = calibration_kkt(make_measurements())
x_test = np.array([.8, .6, .1, -.2])
lam_test = np.array([.3])
gf, J, H, g = sample_kkt(x_test, lam_test)
gf2, J2, H2, g2 = gn_kkt(sample_kkt, lambda z: A)(x_test, lam_test)
assert np.allclose(H2, A.T@A) and np.allclose(gf, gf2) and np.allclose(J, J2) and np.allclose(g, g2)
assert np.linalg.norm(H-H2)>1e-3, "Проверка должна отличать точный и приближённый гессианы"
print("Гаусс–Ньютон: проверка пройдена")

In [ ]:
# TODO: ваш код
# Два значения sigma; запуски Ньютона–Лагранжа и Гаусса–Ньютона; числа, сравнение.
# plot_histories([...], [...], ...) и plot_calibration(measured, z_star)


_Ваш ответ:_ TODO

## Бонус. Сломайте Ньютон (до +2 баллов, 20–30 минут)

Вернитесь к эллипсу задачи 1. Его `ellipse_kkt` дан в начале ноутбука.

**(а) +1 балл.** Запустите `newton_lagrange` из $x_0=(0,0.8)$, $\lambda_0=0.25$ и отдельно из $x_0=(0,-0.8)$, $\lambda_0=-0.25$. Получились ли в обоих случаях **минимумы**? Проверьте знак редуцированного гессиана в ответе.

**(б) +1 балл.** Попробуйте $x_0=(1,0)$, $\lambda_0=0$. Какая именно матрица вырождается? Есть ли при этом LICQ? Докажите вырожденность, не полагаясь только на текст исключения.

Бонус необязателен: оставьте следующие ячейки как есть, если не решаете.

In [ ]:
# Бонус (необязательно): здесь можно добавить ваши эксперименты.
if "bonus_hist" in globals():
    print("Бонус: история длиной", len(bonus_hist))
else:
    print("Бонус пропущен")

_Бонус (необязательно):_ ваш ответ